# 🦉 CalmQuest • Cozy Multi-Agent Stress Monitor (70-Participant GitHub Dataset)
Clones `https://github.com/umarnasir123/stress-dataset-.git`, trains the Random Forest Model Agent, loads `Qwen/Qwen2.5-1.5B-Instruct`, and launches a cozy **Duolingo-style Gradio Live URL** with PDF report generation.

In [ ]:
!pip install -q gradio reportlab scikit-learn matplotlib seaborn pandas numpy transformers accelerate torch openpyxl


In [ ]:
import glob
import os
import pandas as pd

REPO_URL = "https://github.com/umarnasir123/stress-dataset-.git"
REPO_DIR = "stress-dataset-"

# 1. Clone or update the repository
if not os.path.exists(REPO_DIR):
    os.system(f"git clone -q {REPO_URL}")
else:
    os.system(f"git -C {REPO_DIR} pull -q")

# 2. Locate dataset file (.csv, .xlsx, .xls) in the repo
data_files = sorted(
    glob.glob(f"{REPO_DIR}/**/*.csv", recursive=True)
    + glob.glob(f"{REPO_DIR}/**/*.xlsx", recursive=True)
    + glob.glob(f"{REPO_DIR}/**/*.xls", recursive=True)
)
if not data_files:
    raise FileNotFoundError(f"No dataset file found in '{REPO_DIR}'.")

dataset_path = data_files[0]
print("Using dataset file:", dataset_path)

if dataset_path.endswith(".csv"):
    raw_70 = pd.read_csv(dataset_path)
else:
    xls = pd.ExcelFile(dataset_path)
    target_sheet = next(
        (s for s in xls.sheet_names if "participant" in s.lower() or "data" in s.lower()),
        xls.sheet_names[0],
    )
    preview = pd.read_excel(xls, sheet_name=target_sheet, nrows=6)
    has_banner = any("Participant ID" in str(v) for v in preview.iloc[3].values) if len(preview) > 3 else False
    raw_70 = pd.read_excel(
        xls,
        sheet_name=target_sheet,
        header=3 if has_banner else 0,
        nrows=70 if has_banner else None,
    )

raw_70.columns = [str(c).strip() for c in raw_70.columns]
raw_70 = raw_70[raw_70["Participant ID"].astype(str).str.startswith("S")].reset_index(drop=True)

# 3. Build task-level HR, RR, and Stress Label rows for all 70 participants
hr_stress_cols = [c for c in ["Stroop HR (bpm)", "Interview HR (bpm)", "Hyperventilation HR (bpm)"] if c in raw_70.columns]
base_hr_col = "Baseline HR (bpm)" if "Baseline HR (bpm)" in raw_70.columns else raw_70.columns[7]
base_rr_col = "Baseline Resp Rate (brpm)" if "Baseline Resp Rate (brpm)" in raw_70.columns else raw_70.columns[11]
stress_rr_col = "Stress Resp Rate (brpm)" if "Stress Resp Rate (brpm)" in raw_70.columns else raw_70.columns[12]

rows = []
for _, r in raw_70.iterrows():
    pid = str(r["Participant ID"]).strip()
    b_hr = pd.to_numeric(r[base_hr_col], errors="coerce")
    b_rr = pd.to_numeric(r[base_rr_col], errors="coerce")
    s_rr = pd.to_numeric(r[stress_rr_col], errors="coerce")
    if pd.notna(b_hr) and pd.notna(b_rr):
        rows.append({"Participant": pid, "HR": float(b_hr), "RR": float(b_rr), "Label": 0})
    for sc in hr_stress_cols:
        s_hr = pd.to_numeric(r[sc], errors="coerce")
        if pd.notna(s_hr) and pd.notna(s_rr):
            rows.append({"Participant": pid, "HR": float(s_hr), "RR": float(s_rr), "Label": 1})

df = pd.DataFrame(rows).dropna().reset_index(drop=True)
print(f"Loaded {raw_70['Participant ID'].nunique()} participants -> {len(df)} physiological records ready.")


In [ ]:
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer

X_train, X_test, y_train, y_test = train_test_split(
    df[["HR", "RR"]],
    df["Label"],
    test_size=0.2,
    random_state=42,
    stratify=df["Label"],
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
print(f"Model Agent Trained | Test Accuracy: {accuracy_score(y_test, rf_model.predict(X_test))*100:.2f}%")

LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=dtype,
    device_map="auto",
)
print("Clinical Summary Agent (LLM) Ready!")


In [ ]:
from xml.sax.saxutils import escape
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import Image as RLImage, Paragraph, SimpleDocTemplate, Spacer
from sklearn.metrics import confusion_matrix

# Cozy Duolingo-inspired CSS (Warm Cream Canvas, Feather Green 3D Tactile Buttons, Rounded Cards)
DUO_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@600;700;800;900&display=swap');

body, .gradio-container {
    background: linear-gradient(180deg, #FFFDF7 0%, #F4F9F0 100%) !important;
    font-family: 'Nunito', -apple-system, BlinkMacSystemFont, sans-serif !important;
    color: #3C3C3C !important;
}

.duo-hero {
    background: linear-gradient(135deg, #58CC02 0%, #46A302 100%);
    border-radius: 24px;
    padding: 24px 28px;
    color: #FFFFFF !important;
    box-shadow: 0 6px 0 #3B8A00, 0 12px 24px rgba(88, 204, 2, 0.22);
    margin-bottom: 18px;
}
.duo-hero h1 {
    margin: 0 0 6px 0;
    font-size: 28px;
    font-weight: 900;
    letter-spacing: -0.3px;
    color: #FFFFFF !important;
}
.duo-hero p {
    margin: 0;
    font-size: 15px;
    font-weight: 700;
    color: #E9FFD6 !important;
}
.duo-badges {
    display: flex;
    gap: 10px;
    flex-wrap: wrap;
    margin-top: 14px;
}
.duo-badge {
    background: rgba(255, 255, 255, 0.22);
    border: 2px solid rgba(255, 255, 255, 0.45);
    border-radius: 999px;
    padding: 4px 14px;
    font-size: 13px;
    font-weight: 800;
    color: #FFFFFF;
}

.duo-card {
    background: #FFFFFF !important;
    border: 2.5px solid #E5E5E5 !important;
    border-bottom: 5px solid #DCDCDC !important;
    border-radius: 22px !important;
    padding: 18px !important;
    box-shadow: 0 4px 14px rgba(0, 0, 0, 0.03) !important;
}

button.duo-primary-btn {
    background: #58CC02 !important;
    border: none !important;
    border-bottom: 5px solid #46A302 !important;
    border-radius: 18px !important;
    color: #FFFFFF !important;
    font-family: 'Nunito', sans-serif !important;
    font-weight: 900 !important;
    font-size: 17px !important;
    letter-spacing: 0.4px !important;
    padding: 14px 24px !important;
    text-transform: uppercase !important;
    transition: transform 0.08s ease, border-bottom-width 0.08s ease !important;
}
button.duo-primary-btn:hover {
    background: #61E002 !important;
    transform: translateY(-1px) !important;
}
button.duo-primary-btn:active {
    transform: translateY(3px) !important;
    border-bottom-width: 2px !important;
}

.duo-status-normal {
    background: #E9FBE5;
    border: 2.5px solid #58CC02;
    border-bottom: 5px solid #46A302;
    border-radius: 20px;
    padding: 16px 20px;
    color: #2B6300;
    font-weight: 800;
}
.duo-status-alert {
    background: #FFF1F0;
    border: 2.5px solid #FF4B4B;
    border-bottom: 5px solid #D9383A;
    border-radius: 20px;
    padding: 16px 20px;
    color: #9E1C1E;
    font-weight: 800;
}
"""


def _format_pdf_line(line: str) -> str:
    escaped = escape(line.strip())
    parts = escaped.split("**")
    if len(parts) % 2 == 1:
        return "".join(f"<b>{p}</b>" if i % 2 == 1 else p for i, p in enumerate(parts))
    return escaped.replace("**", "")


def run_multi_agent_crew(participant_id, custom_csv, force_pdf):
    if custom_csv is not None:
        u_df = pd.read_csv(custom_csv.name)
        u_df.columns = [c.strip().lower() for c in u_df.columns]
        hr_cols = [c for c in u_df.columns if "hr" in c or "heart" in c]
        rr_cols = [c for c in u_df.columns if "rr" in c or "rsp" in c or "resp" in c]
        if not hr_cols or not rr_cols:
            raise gr.Error("Uploaded CSV must contain HR and RR/respr columns.")
        p_df = pd.DataFrame({
            "HR": pd.to_numeric(u_df[hr_cols[0]], errors="coerce"),
            "RR": pd.to_numeric(u_df[rr_cols[0]], errors="coerce"),
            "Label": pd.to_numeric(u_df["label"], errors="coerce") if "label" in u_df.columns else 0,
        }).dropna().reset_index(drop=True)
        p_df["Label"] = (p_df["Label"] > 0).astype(int)
        subject_tag = "Uploaded_CSV"
        meta_line = "Custom Wearable CSV Upload"
    else:
        p_df = df[df["Participant"] == str(participant_id)].copy().reset_index(drop=True)
        subject_tag = f"Participant_{participant_id}"
        if "raw_70" in globals() and "Participant ID" in raw_70.columns:
            sub = raw_70[raw_70["Participant ID"].astype(str).str.strip() == str(participant_id)]
            if not sub.empty:
                r0 = sub.iloc[0]
                meta_line = f"{r0.get('Assigned Name', participant_id)} • Age {r0.get('Age', 'N/A')} • {r0.get('Gender', '')} • PSS {r0.get('PSS Score (0-40)', 'N/A')}"
            else:
                meta_line = subject_tag
        else:
            meta_line = subject_tag

    if p_df.empty:
        raise gr.Error(f"No valid rows found for {subject_tag}.")

    # --- AGENT 1: MODEL AGENT ---
    p_df["Pred"] = rf_model.predict(p_df[["HR", "RR"]])
    third = max(len(p_df) // 3, 1)
    baseline_pct = p_df["Pred"].iloc[:third].mean() * 100
    dynamic_thresh = max(baseline_pct, 33.3)
    stress_pct = p_df["Pred"].mean() * 100
    is_abnormal = bool(stress_pct > dynamic_thresh)

    # Cozy Duolingo-palette matplotlib charts
    plt.rcParams["font.sans-serif"] = "DejaVu Sans"
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), facecolor="#FFFDF7")
    cm = confusion_matrix(p_df["Label"], p_df["Pred"], labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Greens",
        cbar=False,
        xticklabels=["Calm (0)", "Stress (1)"],
        yticklabels=["True Calm", "True Stress"],
        annot_kws={"size": 13, "weight": "bold"},
        ax=axes[0],
    )
    axes[0].set_title(f"Model Check ({subject_tag})", fontsize=12, fontweight="bold", color="#3C3C3C")

    for idx, (col, unit) in enumerate([("HR", "BPM"), ("RR", "brpm")], start=1):
        axes[idx].set_facecolor("#FFFFFF")
        norm = p_df[p_df["Pred"] == 0]
        strs = p_df[p_df["Pred"] == 1]
        axes[idx].plot(p_df.index, p_df[col], color="#E5E5E5", linewidth=1.8, zorder=1)
        axes[idx].scatter(norm.index, norm[col], c="#58CC02", s=65, edgecolors="#3B8A00", linewidths=1.2, label="Calm Zone (0)", zorder=2)
        axes[idx].scatter(strs.index, strs[col], c="#FF4B4B", s=65, edgecolors="#B81D1F", linewidths=1.2, label="Stress Spike (1)", zorder=3)
        axes[idx].set_title(f"{col} ({unit}) Trajectory", fontsize=12, fontweight="bold", color="#3C3C3C")
        axes[idx].grid(True, linestyle="--", alpha=0.35, color="#CCCCCC")
        axes[idx].spines["top"].set_visible(False)
        axes[idx].spines["right"].set_visible(False)
        axes[idx].legend(frameon=True, facecolor="#FFFDF7", edgecolor="#E5E5E5", fontsize=9)

    plt.tight_layout()
    plot_path = f"agent_signals_{subject_tag}.png"
    plt.savefig(plot_path, dpi=150, facecolor="#FFFDF7")
    plt.close(fig)

    card_cls = "duo-status-alert" if is_abnormal else "duo-status-normal"
    badge_emoji = "🔥" if is_abnormal else "🌿"
    verdict = "HIGH STRESS STREAK DETECTED" if is_abnormal else "CALM & STEADY BASELINE"
    status_html = f"""
    <div class="{card_cls}">
        <div style="font-size:19px; margin-bottom:6px;">{badge_emoji} {verdict} — {escape(subject_tag)}</div>
        <div style="font-size:14px; opacity:0.9;">{escape(meta_line)}</div>
        <div style="display:flex; gap:18px; flex-wrap:wrap; margin-top:10px; font-size:14px;">
            <span>⚡ <b>Stress Load:</b> {stress_pct:.1f}%</span>
            <span>🎯 <b>Personal Threshold:</b> {dynamic_thresh:.1f}%</span>
            <span>❤️ <b>Avg HR:</b> {p_df['HR'].mean():.1f} bpm</span>
            <span>🌬️ <b>Avg RR:</b> {p_df['RR'].mean():.1f} brpm</span>
        </div>
    </div>
    """

    # --- AGENT 2: CLINICAL SUMMARY AGENT ---
    if not is_abnormal and not force_pdf:
        msg = "🌿 Calm streak! Stress stayed below your personal threshold. Toggle 'Always Generate Clinical PDF' to create a report anyway."
        return status_html, plot_path, msg, None

    prompt = (
        f"You are a supportive, clear Clinical Summary AI Coach. Write a structured wellness & clinical report for {subject_tag} ({meta_line}):\n"
        f"- Total Samples: {len(p_df)} | Stressed: {int(p_df['Pred'].sum())} ({stress_pct:.1f}%)\n"
        f"- Dynamic Personal Threshold: {dynamic_thresh:.1f}% | Abnormal Detected: {is_abnormal}\n"
        f"- Mean HR: {p_df['HR'].mean():.1f} BPM | Mean RR: {p_df['RR'].mean():.1f} breaths/min\n"
        "Include 4 clear sections: 1. Clinical Report, 2. Classification Results, 3. Abnormal Patterns, 4. Actionable Wellness Quest (5 numbered points)."
    )
    messages = [{"role": "user", "content": prompt}]
    text_in = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_in], return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=450, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    report_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    pdf_path = f"Clinical_Report_{subject_tag}.pdf"
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [Paragraph(f"<b>Wellness & Clinical Report — {escape(subject_tag)}</b>", styles["Title"]), Spacer(1, 10)]
    for line in report_text.splitlines():
        if line.strip():
            story.append(Paragraph(_format_pdf_line(line), styles["Normal"]))
            story.append(Spacer(1, 5))
    story.extend([Spacer(1, 10), RLImage(plot_path, width=480, height=135)])
    doc.build(story)

    return status_html, plot_path, report_text, pdf_path


pids = sorted(df["Participant"].unique(), key=lambda x: int(x[1:]) if x.startswith("S") and x[1:].isdigit() else 999)
default_pid = "S05" if "S05" in pids else (pids[0] if pids else None)

duo_theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.green,
    secondary_hue=gr.themes.colors.amber,
    neutral_hue=gr.themes.colors.stone,
    radius_size=gr.themes.sizes.radius_lg,
    font=[gr.themes.GoogleFont("Nunito"), "sans-serif"],
)

with gr.Blocks(title="CalmQuest • Multi-Agent Stress Coach") as app:
    gr.HTML("""
    <div class="duo-hero">
        <h1>🦉 CalmQuest • Multi-Agent Stress Monitor</h1>
        <p>Cozy AI Physiological Check-In • 70-Participant Cohort + Autonomous Clinical Coach</p>
        <div class="duo-badges">
            <span class="duo-badge">🔥 70 Subjects Loaded</span>
            <span class="duo-badge">🌲 Random Forest Agent</span>
            <span class="duo-badge">🧠 Qwen2.5 Clinical Coach</span>
            <span class="duo-badge">📄 Instant PDF Export</span>
        </div>
    </div>
    """)

    with gr.Group(elem_classes=["duo-card"]):
        with gr.Row():
            pid_in = gr.Dropdown(
                choices=pids,
                value=default_pid,
                label="🎯 Choose Participant (S01–S70)",
            )
            csv_in = gr.File(
                label="📂 Or Drop Custom Watch CSV (HR, RR)",
                file_types=[".csv"],
            )
            force_in = gr.Checkbox(
                value=True,
                label="✨ Always Generate Clinical PDF",
            )

        run_btn = gr.Button(
            "🚀 Start Stress Check-In",
            elem_classes=["duo-primary-btn"],
        )

    status_out = gr.HTML()

    with gr.Row():
        with gr.Column(scale=3, elem_classes=["duo-card"]):
            img_out = gr.Image(label="📊 Physiological Signals (Green = Calm, Red = Stress)")
        with gr.Column(scale=2, elem_classes=["duo-card"]):
            txt_out = gr.Textbox(label="🦉 Clinical AI Coach Notes", lines=13)
            pdf_out = gr.File(label="📥 Download Clinical PDF Report")

    run_btn.click(
        run_multi_agent_crew,
        inputs=[pid_in, csv_in, force_in],
        outputs=[status_out, img_out, txt_out, pdf_out],
    )

app.launch(share=True, theme=duo_theme, css=DUO_CSS)
